# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Print dataset name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their fields by `@id`

record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in the Croissant metadata. The dataset schema may not explicitly define them or the dataset is comprised of a single default record set.")
    # Try to show files if available
    if hasattr(dataset.metadata, "distribution"):
        print("Distributions found (data files / tables):")
        for dist in dataset.metadata.distribution:
            print(f"  Distribution @id: {getattr(dist, '@id', None)} - name: {getattr(dist, 'name', None)} URL: {getattr(dist, 'contentUrl', None)}")
    else:
        print("No file distributions available for exploration.")
else:
    print("Record sets and their fields:")
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']} - name: {rs.get('name', None)}")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            print(f"  \u2514 Field @id: {field['@id']} - name: {field.get('name', None)}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Since the dataset does not define explicit record sets, we load the default data table records.
# By convention with mlcroissant, we can use None or fetch distribution @id directly if needed.

dfs = {}

# Helper: List available data tables (distributions considered as tables)
data_distributions = []
if hasattr(dataset.metadata, "distribution"):
    for dist in dataset.metadata.distribution:
        if hasattr(dist, 'encodingFormat') and (
            ('csv' in str(dist.encodingFormat).lower()) or ('tsv' in str(dist.encodingFormat).lower())
        ):
            data_distributions.append(dist)
if not data_distributions:
    print("No CSV/TSV data distributions available for automatic table loading.")
else:
    print(f"Found {len(data_distributions)} data distribution(s)")
    for i, dist in enumerate(data_distributions):
        print(f"[{i}] Distribution @id: {dist['@id']}, Name: {getattr(dist, 'name', None)}, Format: {getattr(dist, 'encodingFormat', None)}")

    # Use the first as an example
    distribution_id = data_distributions[0]['@id']

    records = list(dataset.records(record_set=distribution_id))
    df = pd.DataFrame(records)
    dfs[distribution_id] = df
    print(f"Loaded DataFrame for Distribution @id: {distribution_id}")
    print("Columns:", df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# --- Example: Numeric Field Analysis on Protein Dataset ---
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Pick first data table and a likely numeric field by name
if dfs:
    # For this dataset, likely fields include: 'Coverage(%)', 'MW', 'Peptides', etc.
    df = list(dfs.values())[0]
    numeric_field_candidates = [c for c in df.columns if any(k in c.lower() for k in ["coverage", "mw", "peptide", "abundance", "count", "intensity", "score", "%"])]
    print('Numeric candidates:', numeric_field_candidates)

    # Try with 'Coverage(%)' if it exists
    numeric_field = None
    for n in ['Coverage(%)', 'Coverage', 'MW', 'Peptides']:
        if n in df.columns:
            numeric_field = n
            break
    if numeric_field is None and numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]

    if numeric_field is not None:
        print(f"Analyzing field: {numeric_field}")
        # Convert to numeric (when reading from CSV, may be string)
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = float(np.percentile(df[numeric_field].dropna(), 75))
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (top quartile):")
        display(filtered_df.head())
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())
        # Try grouping by protein description if present
        group_field = None
        for g in ['Description', 'Protein description', 'Gene', 'Accession']:
            if g in df.columns:
                group_field = g
                break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (showing mean of {numeric_field}):")
            display(grouped_df.head())
    else:
        print("No suitable numeric field found for EDA.")
else:
    print('No DataFrames loaded for analysis.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dfs and numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If there are at least two numeric fields, plot scatter
    if len(numeric_field_candidates) >= 2:
        xfield = numeric_field_candidates[0]
        yfield = numeric_field_candidates[1]
        plt.figure(figsize=(6, 5))
        plt.scatter(df[xfield], df[yfield], alpha=0.7)
        plt.xlabel(xfield)
        plt.ylabel(yfield)
        plt.title(f"Scatter plot: {xfield} vs. {yfield}")
        plt.show()
else:
    print('No suitable numeric field(s) for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook has demonstrated how to use the `mlcroissant` library to access FAIR-compliant tabular life science datasets described by a Croissant schema.

- We loaded the dataset metadata and explored available data tables and their structure.
- Numeric data fields such as protein coverage or molecular weight were automatically detected and analyzed.
- Data was filtered, normalized, grouped, and visualized, showcasing typical EDA workflows for proteomics tables.

For detailed downstream analyses, continue with domain-specific statistical models, visualization, or integration with biological databases using field `@id` links and Croissant schema metadata.